In [4]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import time
import json
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score

In [5]:
BASE_DIR = r"E:\SmartVision-AI\data"
CLASSIFICATION_DIR = os.path.join(BASE_DIR, "classification")

TRAIN_DIR = os.path.join(CLASSIFICATION_DIR, "train")
VAL_DIR = os.path.join(CLASSIFICATION_DIR, "val")
TEST_DIR = os.path.join(CLASSIFICATION_DIR, "test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)
print(torch.cuda.get_device_name(0))

Using device: cuda
NVIDIA GeForce RTX 4050 Laptop GPU


In [6]:
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", num_classes)

Classes: 26


In [8]:
model = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features

model.fc = nn.Sequential(
    nn.Linear(num_features, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, num_classes)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

print(model.fc)

Sequential(
  (0): Linear(in_features=2048, out_features=512, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=512, out_features=26, bias=True)
)


In [10]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    best_val_acc = 0.0

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(epochs):

        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 30)

        model.train()

        train_loss = 0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)

            train_loss += loss.item() * images.size(0)
            train_correct += torch.sum(preds == labels).item()
            train_total += labels.size(0)

        train_epoch_loss = train_loss / train_total
        train_epoch_acc = train_correct / train_total

        model.eval()

        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

                val_loss += loss.item() * images.size(0)
                val_correct += torch.sum(preds == labels).item()
                val_total += labels.size(0)

        val_epoch_loss = val_loss / val_total
        val_epoch_acc = val_correct / val_total

        history["train_loss"].append(train_epoch_loss)
        history["train_acc"].append(train_epoch_acc)
        history["val_loss"].append(val_epoch_loss)
        history["val_acc"].append(val_epoch_acc)

        print(f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_epoch_acc:.4f}")
        print(f"Val Loss:   {val_epoch_loss:.4f} | Val Acc:   {val_epoch_acc:.4f}")

        if val_epoch_acc > best_val_acc:

            best_val_acc = val_epoch_acc

            torch.save(
                model.state_dict(),
                r"E:\SmartVision-AI\models\resnet50_best.pth"
            )

            print("✅ Best model saved")

    return history

In [11]:
start_time = time.time()

history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=10
)

end_time = time.time()
training_time = end_time - start_time

print(f"Training Time: {training_time/60:.2f} minutes")


Epoch 1/10
------------------------------
Train Loss: 2.5545 | Train Acc: 0.3253
Val Loss:   1.9395 | Val Acc:   0.5205
✅ Best model saved

Epoch 2/10
------------------------------
Train Loss: 1.5660 | Train Acc: 0.5648
Val Loss:   1.3836 | Val Acc:   0.6205
✅ Best model saved

Epoch 3/10
------------------------------
Train Loss: 1.2177 | Train Acc: 0.6753
Val Loss:   1.1903 | Val Acc:   0.6744
✅ Best model saved

Epoch 4/10
------------------------------
Train Loss: 1.0086 | Train Acc: 0.7143
Val Loss:   1.0894 | Val Acc:   0.6744

Epoch 5/10
------------------------------
Train Loss: 0.9147 | Train Acc: 0.7456
Val Loss:   1.0399 | Val Acc:   0.7359
✅ Best model saved

Epoch 6/10
------------------------------
Train Loss: 0.7603 | Train Acc: 0.7857
Val Loss:   0.9846 | Val Acc:   0.7333

Epoch 7/10
------------------------------
Train Loss: 0.7404 | Train Acc: 0.7945
Val Loss:   0.9985 | Val Acc:   0.7308

Epoch 8/10
------------------------------
Train Loss: 0.6488 | Train Acc: 0.

In [12]:
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)

history_finetune = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    epochs=5
)


Epoch 1/5
------------------------------
Train Loss: 0.4315 | Train Acc: 0.8780
Val Loss:   0.8487 | Val Acc:   0.7641
✅ Best model saved

Epoch 2/5
------------------------------
Train Loss: 0.3306 | Train Acc: 0.9000
Val Loss:   0.8585 | Val Acc:   0.7744
✅ Best model saved

Epoch 3/5
------------------------------
Train Loss: 0.2541 | Train Acc: 0.9319
Val Loss:   0.8713 | Val Acc:   0.7718

Epoch 4/5
------------------------------
Train Loss: 0.2220 | Train Acc: 0.9374
Val Loss:   0.8804 | Val Acc:   0.7795
✅ Best model saved

Epoch 5/5
------------------------------
Train Loss: 0.2021 | Train Acc: 0.9423
Val Loss:   0.8889 | Val Acc:   0.7974
✅ Best model saved


In [13]:
model.load_state_dict(torch.load(r"E:\SmartVision-AI\models\resnet50_best.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)

print("ResNet50 Test Accuracy:", round(test_acc, 4))
print()
print(classification_report(all_labels, all_preds, target_names=class_names))

ResNet50 Test Accuracy: 0.7846

               precision    recall  f1-score   support

     airplane       0.93      0.87      0.90        15
          bed       0.67      0.40      0.50        15
        bench       0.69      0.73      0.71        15
      bicycle       0.88      0.93      0.90        15
         bird       1.00      0.93      0.97        15
       bottle       0.88      0.93      0.90        15
         bowl       0.65      0.73      0.69        15
          bus       0.86      0.80      0.83        15
         cake       1.00      0.80      0.89        15
          car       0.74      0.93      0.82        15
          cat       0.79      0.73      0.76        15
        chair       0.80      0.80      0.80        15
        couch       0.56      0.67      0.61        15
          cow       0.94      1.00      0.97        15
          cup       0.55      0.73      0.63        15
          dog       0.67      0.67      0.67        15
     elephant       0.93      0.

In [14]:
resnet50_results = {
    "model": "ResNet50",
    "test_accuracy": 0.7846,
    "precision_macro": 0.80,
    "recall_macro": 0.78,
    "macro_f1": 0.79,
    "num_classes": num_classes
}

with open(r"E:\SmartVision-AI\models\resnet50_metrics.json", "w") as f:
    json.dump(resnet50_results, f, indent=4)

print("ResNet50 metrics saved")

ResNet50 metrics saved
